# Finite Difference Method (FDM)

### Black & Scholes

In [1]:
import numpy as np
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

### Explicit Scheme

In [2]:
import numpy as np

def explicit_fdm(S0, K, T, r, sigma, Smax, N, M, option_type='call'):
    dS, dt = Smax / N, T / M
    i = np.arange(0, N + 1)
    S = i * dS
    V = np.maximum(S - K, 0.0) if option_type == 'call' else np.maximum(K - S, 0.0)

    # Coefficients a_i, b_i, c_i
    a = 0.5 * dt * (sigma**2 * i**2 - r * i)
    b = 1 - dt * (sigma**2 * i**2 + r)
    c = 0.5 * dt * (sigma**2 * i**2 + r * i)

    for j in range(M, 0, -1):
        # Vectorized across all interior nodes at once
        V_new = V.copy()
        V_new[1:N] = a[1:N] * V[0:N-1] + b[1:N] * V[1:N] + c[1:N] * V[2:N+1]
        
        # Boundary condition
        tau = T - (j - 1) * dt
        V_new[0], V_new[N] = (0.0, Smax - K * np.exp(-r * tau)) if option_type == 'call' else (K * np.exp(-r * tau), 0.0)
        
        V = V_new

    # Linear interpolation to read the price at S0
    idx = int(S0 / dS)
    price = V[idx] + (S0 - S[idx]) / dS * (V[idx+1] - V[idx])
    return price

In [3]:
# Parameters
S0, K, T, r, sigma, Smax = 100.0, 100.0, 1.0, 0.05, 0.20, 300.0

In [4]:
# Pricing
for M in [20, 50, 100, 1000, 3000]:
    ex = explicit_fdm(S0, K, T, r, sigma, Smax, 120, M, 'put')
    print(f"N={120} | M={M:5d} (dt={T/M:.5f}) | Explicit={ex:>24,.2f}")

N=120 | M=   20 (dt=0.05000) | Explicit=  -56,806,074,026,675.11
N=120 | M=   50 (dt=0.02000) | Explicit= -319,074,587,115,928.62
N=120 | M=  100 (dt=0.01000) | Explicit=          -79,124,686.79
N=120 | M= 1000 (dt=0.00100) | Explicit=                    5.56
N=120 | M= 3000 (dt=0.00033) | Explicit=                    5.56


### Implicit Scheme

In [5]:
from scipy.linalg import solve_banded

def implicit_fdm(S0, K, T, r, sigma, Smax, N, M, option_type='call'):
    dS, dt = Smax / N, T / M
    i = np.arange(0, N + 1)
    S = i * dS
    V = np.maximum(S - K, 0.0) if option_type == 'call' else np.maximum(K - S, 0.0)

    # Coefficients alpha_i, beta_i, gamma_i
    alpha = 0.5 * dt * (r * i - sigma**2 * i**2)
    beta  = 1 + dt * (sigma**2 * i**2 + r)
    gamma = -0.5 * dt * (r * i + sigma**2 * i**2)

    # Tridiagonal matrix
    A = np.zeros((3, N - 1))
    A[0, 1:]  = gamma[1:N-1]   # superdiagonal
    A[1, :]   = beta[1:N]      # main diagonal
    A[2, :-1] = alpha[2:N]     # subdiagonal

    for j in range(M, 0, -1):
        # Boundary condition
        tau = T - (j - 1) * dt
        V0, VN = (0.0, Smax - K * np.exp(-r * tau)) if option_type == 'call' else (K * np.exp(-r * tau), 0.0)
        
        d = V[1:N].copy()
        d[0]  -= alpha[1] * V0
        d[-1] -= gamma[N-1] * VN
        x = solve_banded((1, 1), A, d) #Thomas algorithm
        V = np.concatenate(([V0], x, [VN]))

    idx = int(S0 / dS)
    price = V[idx] + (S0 - S[idx]) / dS * (V[idx+1] - V[idx])
    return price

In [6]:
# Pricing
for M in [20, 50, 100]:
    ex = explicit_fdm(S0, K, T, r, sigma, Smax, 120, M, 'put')
    im = implicit_fdm(S0, K, T, r, sigma, Smax, 120, M, 'put')
    print(f"N={120} | M={M:3d} (dt={T/M:.5f}) | Explicit={ex:>24,.2f} | Implicit={im:>5,.2f}")

N=120 | M= 20 (dt=0.05000) | Explicit=  -56,806,074,026,675.11 | Implicit= 5.51
N=120 | M= 50 (dt=0.02000) | Explicit= -319,074,587,115,928.62 | Implicit= 5.54
N=120 | M=100 (dt=0.01000) | Explicit=          -79,124,686.79 | Implicit= 5.55


### Crank-Nicolson Scheme

In [7]:
def crank_nicolson_fdm(S0, K, T, r, sigma, Smax, N, M, option_type='call'):
    dS, dt = Smax / N, T / M
    i = np.arange(0, N + 1)
    S = i * dS
    V = np.maximum(S - K, 0.0) if option_type == 'call' else np.maximum(K - S, 0.0)

    a = 0.25 * dt * (sigma**2 * i**2 - r * i) # a = a_explicit/2
    b = -0.5 * dt * (sigma**2 * i**2 + r)     # b = (b_explicit - 1)/2
    c = 0.25 * dt * (sigma**2 * i**2 + r * i) # c = c_explicit/2

    A = np.zeros((3, N - 1))
    A[0, 1:]  = -c[1:N-1]
    A[1, :]   = 1 - b[1:N]
    A[2, :-1] = -a[2:N]

    for j in range(M, 0, -1):
        # Boundary condition
        tau = T - (j - 1) * dt
        V0, VN = (0.0, Smax - K * np.exp(-r * tau)) if option_type == 'call' else (K * np.exp(-r * tau), 0.0)
        
        d = a[1:N] * V[0:N-1] + (1 + b[1:N]) * V[1:N] + c[1:N] * V[2:N+1]
        d[0]  += c[1] * V0
        d[-1] += a[N-1] * VN
        x = solve_banded((1, 1), A, d)
        V = np.concatenate(([V0], x, [VN]))

    idx = int(S0 / dS)
    price = V[idx] + (S0 - S[idx]) / dS * (V[idx+1] - V[idx])
    return price

In [8]:
# Pricing
bs_price = black_scholes(S0, K, T, r, sigma, 'put')

for N, M in [(60, 500), (60, 1000), (120, 1000), (120, 1500), (200, 1500)]:
    ex  = explicit_fdm(S0, K, T, r, sigma, Smax, N, M, 'put')
    im = implicit_fdm(S0, K, T, r, sigma, Smax, N, M, 'put')
    cn = crank_nicolson_fdm(S0, K, T, r, sigma, Smax, N, M, 'put')
    print(f"N={N:3d} M={M:4d} | Expl={ex:.4f} (err={abs(ex-bs_price):.4f}) | "
          f"Impl={im:.4f} (err={abs(im-bs_price):.4f}) | CN={cn:.4f} (err={abs(cn-bs_price):.4f})")

N= 60 M= 500 | Expl=5.5130 (err=0.0606) | Impl=5.5091 (err=0.0644) | CN=5.5110 (err=0.0625)
N= 60 M=1000 | Expl=5.5120 (err=0.0615) | Impl=5.5101 (err=0.0634) | CN=5.5110 (err=0.0625)
N=120 M=1000 | Expl=5.5590 (err=0.0145) | Impl=5.5571 (err=0.0164) | CN=5.5580 (err=0.0155)
N=120 M=1500 | Expl=5.5587 (err=0.0149) | Impl=5.5574 (err=0.0161) | CN=5.5580 (err=0.0155)
N=200 M=1500 | Expl=5.5780 (err=0.0045) | Impl=5.5768 (err=0.0032) | CN=5.5774 (err=0.0038)


In [9]:
for M in [20, 50, 100]:
    ex  = explicit_fdm(S0, K, T, r, sigma, Smax, 120, M, 'put')
    im = implicit_fdm(S0, K, T, r, sigma, Smax, 120, M, 'put')
    cn = crank_nicolson_fdm(S0, K, T, r, sigma, Smax, 120, M, 'put')
    print(f"M={M:3d} (dt={T/M:.5f}) | Explicit={ex:>24,.2f} | Implicit={im:.4f} | CN={cn:.4f}")

M= 20 (dt=0.05000) | Explicit=  -56,806,074,026,675.11 | Implicit=5.5114 | CN=5.5587
M= 50 (dt=0.02000) | Explicit= -319,074,587,115,928.62 | Implicit=5.5393 | CN=5.5581
M=100 (dt=0.01000) | Explicit=          -79,124,686.79 | Implicit=5.5487 | CN=5.5581
